# Part 6 — ML Prediction Demonstration
## Smart Healthcare Assistant and Hospital Management System
### Predicting AI-Assisted HMS Adoption Readiness in Bangladesh

**Continuation of Part 5.** Part 5 built and evaluated several machine-learning models
(Dummy baseline, Logistic Regression, Decision Tree, Random Forest) on the synthetic
`synthetic_hms_adoption_dataset.csv` dataset and selected **Logistic Regression** as the
best-performing model (by Macro F1 / Balanced Accuracy).

This notebook does **not** redesign, retune, or re-evaluate the modeling from Part 5. It simply
demonstrates, end-to-end, how the trained Logistic Regression model can be applied to a **new**
person's information to predict their AI-assisted HMS **adoption readiness category**
(`Low`, `Medium`, or `High`).

## Section 1 — Introduction

**What this notebook does:**
- Part 5 developed and evaluated multiple ML models and selected Logistic Regression as the
  best-performing model on the synthetic dataset.
- Part 6 (this notebook) demonstrates how that same, unmodified Logistic Regression pipeline
  from Part 5 can be used to generate a prediction for a **new** healthcare worker/hospital
  context that is not in the original dataset.
- The model is retrained here directly from the original CSV, using the exact same feature
  columns, preprocessing, and Logistic Regression configuration as Part 5 — nothing about the
  methodology has changed.

**Important note on the data:** `synthetic_hms_adoption_dataset.csv` is a **synthetic /
simulated** dataset. It was not collected from real healthcare workers or hospitals. Therefore,
**this demonstration is not evidence of real-world predictive performance** — it only shows how
the prediction *mechanism* would work if applied to real, validated data in the future.

## Section 2 — Load Dataset and Recreate the Part 5 Model

We load the original CSV exactly as provided and rebuild the **same** preprocessing pipeline and
Logistic Regression model used in Part 5:

- Feature columns, categorical/numerical split — identical to Part 5.
- Preprocessing — `OneHotEncoder(handle_unknown="ignore")` for categorical features,
  `StandardScaler()` for numerical features, combined with `ColumnTransformer`.
- Train/test split — 80/20, stratified by target, `random_state=42`.
- Model — `LogisticRegression(max_iter=1000, random_state=42)`.

No model coefficients or results are typed in manually — everything below is computed from the
CSV.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# sklearn: splitting, preprocessing, model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

# Load the ORIGINAL uploaded CSV exactly as-is. Do not modify or regenerate it.
DATA_PATH = "synthetic_hms_adoption_dataset.csv"   # update path if needed
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Same feature columns, target, and categorical/numerical split as Part 5
feature_columns = [
    "age",
    "gender",
    "profession",
    "education_level",
    "years_of_experience",
    "hospital_type",
    "hospital_location",
    "ai_awareness_score",
    "privacy_score",
    "human_factor_score",
    "infrastructure_score",
]

target_column = "adoption_readiness_category"

categorical_features = ["gender", "profession", "education_level", "hospital_type", "hospital_location"]
numerical_features = ["age", "years_of_experience", "ai_awareness_score",
                       "privacy_score", "human_factor_score", "infrastructure_score"]

assert set(categorical_features + numerical_features) == set(feature_columns)
assert all(col in df.columns for col in feature_columns + [target_column])

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Same 80/20 stratified train/test split as Part 5
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

In [ ]:
# Same preprocessing pipeline used for Logistic Regression in Part 5
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features),
    ]
)

# Same Logistic Regression configuration selected as the best model in Part 5
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

logreg_pipeline.fit(X_train, y_train)
y_pred_logreg = logreg_pipeline.predict(X_test)

print("Logistic Regression model retrained (same configuration as Part 5).")
print("Test accuracy:", round(accuracy_score(y_test, y_pred_logreg), 3))

# The class order the model works with
class_order = list(logreg_pipeline.named_steps["classifier"].classes_)
print("Model classes:", class_order)

## Section 3 — New Person Input

Enter information for a **new** person below. This is a simple set of Python variables — no web
application is needed. Change the values to describe any hypothetical person.

**Fields:**
1. `age` — years
2. `gender` — one of: `Male`, `Female`, `Other/Prefer not to say`
3. `profession` — one of: `Doctor`, `Nurse`, `Hospital Administrator`, `IT Staff`,
   `Healthcare Student`, `Other`
4. `education_level` — one of: `Secondary/HSC`, `Bachelor's (ongoing)`, `Bachelor's (completed)`,
   `Postgraduate/Doctoral`
5. `years_of_experience` — years
6. `hospital_type` — one of: `Public`, `Private`, `NGO/Other`, `Not Applicable`
7. `hospital_location` — one of: `Dhaka`, `Chattogram`, `Other Urban`, `Semi-Urban`, `Rural`,
   `Not Applicable`
8. `ai_awareness_score` — 1 to 5
9. `privacy_score` — 1 to 5 (privacy concern)
10. `human_factor_score` — 1 to 5 (human-factor concern)
11. `infrastructure_score` — 1 to 5 (infrastructure readiness)

In [ ]:
# Edit these values to describe a new person
new_person = {
    "age": 34,
    "gender": "Female",
    "profession": "Nurse",
    "education_level": "Bachelor's (completed)",
    "years_of_experience": 8,
    "hospital_type": "Public",
    "hospital_location": "Dhaka",
    "ai_awareness_score": 4,
    "privacy_score": 2,
    "human_factor_score": 3,
    "infrastructure_score": 4,
}

# Convert to a one-row DataFrame with the exact same column order as training
new_person_df = pd.DataFrame([new_person])[feature_columns]
new_person_df

## Section 4 — Make Prediction

The new person's information is passed through the **same fitted pipeline** (preprocessing +
Logistic Regression) trained in Section 2. The predicted class comes directly from the model —
it is not manually assigned.

In [ ]:
predicted_class = logreg_pipeline.predict(new_person_df)[0]

print(f"Predicted Adoption Readiness: {predicted_class.upper()}")

## Section 5 — Prediction Probabilities

The model's probability for each class, computed with `predict_proba()`.

In [ ]:
proba = logreg_pipeline.predict_proba(new_person_df)[0]

proba_df = pd.DataFrame({
    "Class": logreg_pipeline.named_steps["classifier"].classes_,
    "Probability (%)": (proba * 100).round(1)
}).sort_values("Class").reset_index(drop=True)

print("Prediction Probabilities")
for _, row in proba_df.iterrows():
    print(f"{row['Class']:<8} {row['Probability (%)']:>5.1f}%")

proba_df

## Section 6 — Simple Interpretation

A short, automatically generated interpretation based on the predicted class. This describes what
the *model* predicts — it is **not** a diagnosis or a definitive judgment about the person.

In [ ]:
interpretation_map = {
    "Low": "The model predicts low readiness for adopting an AI-assisted HMS based on the provided characteristics.",
    "Medium": "The model predicts moderate readiness for adopting an AI-assisted HMS based on the provided characteristics.",
    "High": "The model predicts high readiness for adopting an AI-assisted HMS based on the provided characteristics.",
}

print(interpretation_map[predicted_class])
print("\nNote: this is a model-generated prediction based on a synthetic dataset, not a diagnosis")
print("or a definitive judgment about this or any real person.")

## Section 7 — Try Multiple Example Persons

Three example profiles are run through the same trained model. Their predictions are generated
naturally by the model — they are **not** forced to be Low/Medium/High, and it is acceptable if
two examples receive the same prediction.

- **Example 1:** relatively low AI awareness / readiness indicators
- **Example 2:** moderate readiness indicators
- **Example 3:** relatively high AI awareness / readiness indicators

In [ ]:
example_persons = {
    "Example 1": {
        "age": 45,
        "gender": "Male",
        "profession": "Other",
        "education_level": "Secondary/HSC",
        "years_of_experience": 20,
        "hospital_type": "Public",
        "hospital_location": "Rural",
        "ai_awareness_score": 1,
        "privacy_score": 5,
        "human_factor_score": 5,
        "infrastructure_score": 1,
    },
    "Example 2": {
        "age": 30,
        "gender": "Female",
        "profession": "Nurse",
        "education_level": "Bachelor's (completed)",
        "years_of_experience": 6,
        "hospital_type": "Private",
        "hospital_location": "Semi-Urban",
        "ai_awareness_score": 3,
        "privacy_score": 3,
        "human_factor_score": 3,
        "infrastructure_score": 3,
    },
    "Example 3": {
        "age": 27,
        "gender": "Male",
        "profession": "IT Staff",
        "education_level": "Postgraduate/Doctoral",
        "years_of_experience": 3,
        "hospital_type": "Private",
        "hospital_location": "Dhaka",
        "ai_awareness_score": 5,
        "privacy_score": 1,
        "human_factor_score": 1,
        "infrastructure_score": 5,
    },
}

examples_df = pd.DataFrame(example_persons).T[feature_columns]

example_predictions = logreg_pipeline.predict(examples_df)
example_proba = logreg_pipeline.predict_proba(examples_df)

comparison_table = pd.DataFrame({
    "Example": examples_df.index,
    "Predicted Readiness": example_predictions,
})

for i, cls in enumerate(logreg_pipeline.named_steps["classifier"].classes_):
    comparison_table[f"P({cls}) %"] = (example_proba[:, i] * 100).round(1)

comparison_table = comparison_table.reset_index(drop=True)
comparison_table

In [ ]:
print("Example        Predicted Readiness")
for ex, pred in zip(comparison_table["Example"], comparison_table["Predicted Readiness"]):
    print(f"{ex:<14} {pred}")

## Section 8 — Limitations

- The dataset (`synthetic_hms_adoption_dataset.csv`) is **synthetic** — it was generated to
  simulate plausible patterns, not collected from real healthcare workers or hospitals.
- The Logistic Regression model used in this notebook was trained entirely on this synthetic
  data, using the same methodology validated in Part 5.
- Predictions produced above (for the new person and the three example persons) should **not**
  be interpreted as validated predictions of real healthcare workers' or hospitals' actual
  adoption readiness.
- Real-world deployment of a prediction tool like this would require **real, ethically
  collected, representative** healthcare data, along with re-validation of the model on that
  data before any operational use.
- This notebook is intended only as a **demonstration** of how the ML prediction component of
  the project could work in practice, using the same modeling approach finalized in Part 5.

> **Part 6 demonstrates the prediction mechanism only. It does not provide, and should not be
> interpreted as providing, evidence of real-world predictive performance among Bangladeshi
> healthcare workers or hospitals.**